# WDLadder: scaled weight decay on an MAI-style MoE scaling ladder (orchestrator)
Run this notebook on the **compute instance `cpurlvr`** (kernel *Python 3.10 - SDK v2*), one cell at a time.
Every step ends with **CHECK BEFORE CONTINUING**. If a check fails, stop and fix it, because later steps cost money.

| Step | Where | Cost |
|---|---|---|
| 0-4 setup, unit + local smoke tests | instance | ~$1 |
| 5-6 full data pool + upload | instance | ~$3 |
| 7 cluster smoke (pre-emption/resume on CPU + GPU) | clusters | ~$0.3 |
| 8 benchmarks -> real budget table | clusters | ~$0.5 |
| 9-10 stage 1 -> **GATE 1** | CPU cluster | ~$4 |
| 11-14 stages 2-6 (each gated) | CPU + T4 | ~$40-135 |

**Stop the compute instance whenever you are not using it** ($1.31/h). Cluster jobs keep running without it.
Use the check-in cell (9b) once or twice a day to resubmit pre-empted runs; each check-in takes about 1 minute.

In [1]:
# ===== H (v2) =====
import os, sys, json, glob, shutil, importlib, importlib.util, nbformat
from IPython import get_ipython
W = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder"
ALT = "/home/azureuser/cloudfiles/code/Users/estherxin0011/wdladder"
os.chdir(W); sys.path.insert(0, W); importlib.invalidate_caches()
if not os.path.exists(W + "/chain.py"):
    src = next((p for d in (ALT, W, os.path.dirname(ALT)) for p in sorted(glob.glob(d + "/*chain*.py"))), None)
    assert src, "chain.py not found in " + ALT + " or " + W
    shutil.copy(src, W + "/chain.py"); print("copied", src, "->", W + "/chain.py")
import config as C, azure_jobs as AJ
spec = importlib.util.spec_from_file_location("chain", W + "/chain.py")
chain = importlib.util.module_from_spec(spec); sys.modules["chain"] = chain; spec.loader.exec_module(chain)
print("chain loaded:", chain.__file__, os.path.getsize(chain.__file__), "bytes")
PY = sys.executable
def NB(key, i=None):
    nb = nbformat.read("orchestrate.ipynb", 4); md = ""; hits = []
    for c in nb.cells:
        if c.cell_type == "markdown": md = c.source; continue
        if key.lower() in (md + "\n" + c.source).lower(): hits.append(c.source)
    if not hits: return print("no cell matches", key)
    if i is None and len(hits) > 1:
        for k, s in enumerate(hits): print(f"--- [{k}]\n{s[:300]}\n")
        return print(f"{len(hits)} matches -> NB({key!r}, i=K)")
    get_ipython().run_cell(hits[i or 0])
def DONE(stages, l6_tpp=120):
    st = {r["name"]: chain._remote_state(r["name"]) for r in AJ.stage_runs(stages, l6_tpp)}
    print({s: list(st.values()).count(s) for s in set(st.values())}); return st



chain loaded: /mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/chain.py 18807 bytes


In [2]:
import glob, json, re, shutil, numpy as np
R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
AJ.sync(); os.makedirs(X, exist_ok=True)
for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
    shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
pat = re.compile(r"^L(\d+)_t120_cos_(const|scaled)_lr([\d.]+)_p([\d.]+)_s(\d+)$")
D = {}
for d in glob.glob(R + "/L*_t120_cos_*"):
    m = pat.match(os.path.basename(d))
    if not m or float(m[3]) != 1.0: continue
    try: s = json.load(open(d + "/summary.json"))
    except Exception: continue
    if s.get("status") == "done":
        D.setdefault((int(m[1]), m[2]), {}).setdefault(float(m[4]), []).append(s["final"]["bpb"])

def fit(pts):
    ps = sorted(pts); x = np.log2(ps); y = np.array([np.mean(pts[p]) for p in ps])
    if len(ps) >= 3:
        k = np.polyfit(x, y, 2); a, b = k[0], k[1]
        xm = float(np.clip(-b / (2 * a), x.min() - 1, x.max() + 1)) if a > 0 else float(x[y.argmin()])
        ymin = min(float(np.polyval(k, xm)), float(y.min()))
        y1 = float(np.mean(pts[1.0])) if 1.0 in pts else float(np.polyval(k, 0.0))
    else:
        xm, ymin = float(x[y.argmin()]), float(y.min())
        y1 = float(np.mean(pts[1.0])) if 1.0 in pts else float(np.interp(0.0, x, y))
    edge = not (x.min() <= xm <= x.max())
    return dict(n=len(ps), p_opt=round(2 ** xm, 3), bpb_opt=ymin, bpb_p1=y1, edge=edge)

print(f"{'L':>3} {'untuned gap':>12} {'tuned gap':>10} {'survival':>9} {'p_opt c/s':>16} {'WD ratio':>9}  edge(c,s)  npts(c,s)")
for L in sorted({k[0] for k in D}):
    if (L, "const") not in D or (L, "scaled") not in D: continue
    c, s = fit(D[(L, "const")]), fit(D[(L, "scaled")])
    ug, tg = c["bpb_p1"] - s["bpb_p1"], c["bpb_opt"] - s["bpb_opt"]
    print(f"{L:>3} {ug:12.5f} {tg:10.5f} {tg/ug if ug > 0 else float('nan'):9.3f} "
          f"{c['p_opt']:>7}/{s['p_opt']:<7} {s['p_opt']/c['p_opt']:9.2f}  {c['edge']!s:5},{s['edge']!s:5}  {c['n']},{s['n']}")
for k in sorted(D): print(k, {p: round(float(np.mean(v)), 5) for p, v in sorted(D[k].items())})

synced 434 files -> /home/azureuser/wdladder_data/runs
  L  untuned gap  tuned gap  survival        p_opt c/s  WD ratio  edge(c,s)  npts(c,s)
  3      0.03499    0.00360     0.103   0.192/0.269        1.40  False,False  6,7
  4      0.03105    0.00402     0.130   0.253/0.414        1.64  False,True   5,5
  6      0.02407    0.00513     0.213   0.331/0.632        1.91  True ,True   4,4
(3, 'const') {0.125: 1.27778, 0.25: 1.27925, 0.5: 1.29115, 1.0: 1.32209, 2.0: 1.37421, 4.0: 1.43913}
(3, 'scaled') {0.125: 1.27648, 0.25: 1.27474, 0.5: 1.27589, 1.0: 1.2871, 2.0: 1.31112, 4.0: 1.34722, 8.0: 1.38583}
(4, 'const') {0.25: 1.18881, 0.5: 1.19805, 1.0: 1.22397, 2.0: 1.26833, 4.0: 1.331}
(4, 'scaled') {0.5: 1.18576, 1.0: 1.19292, 2.0: 1.21365, 4.0: 1.24669, 8.0: 1.28723}
(6, 'const') {0.354: 1.02102, 0.707: 1.03145, 1.41: 1.05772, 2.83: 1.1022}
(6, 'scaled') {0.708: 1.01592, 1.41: 1.02416, 2.83: 1.0431, 5.66: 1.07497}


In [3]:
!{PY} analysis.py --stage 5
!{PY} llm_review.py review --stage 5
chain.stop("gpu"); chain.status("gpu", runs=False)

runs loaded: 81 (81 done)
GATE 1: PASS {'gain_mean': 0.03499458361150165, 'threshold': 0.003, 'speedup_untuned': 1.5183786483006478}
edge_extension : []
wd_ratio_cos : 1.4080200039912556
tuned_gap : {"gap": 0.002583278761996155, "ci": [0.0010553279314982537, 0.0041112295924940565], "const": [1.2786254515921505, 1.278447403896516], "scaled": [1.2756821067972361, 1.276224191167438]}
survival_fraction : 0.07381938847093783
hypothesis_verdict : {"cos": {"measured": 1.4080200039912556, "ci": [0.877382105149401, 2.254021055010117], "consistent": ["H_none", "H_int"], "closest": "H_int"}, "wsd": {"measured": 0.28702679510801843, "ci": [0.19135119673867895, 0.43054019266202764], "consistent": [], "closest": "H_none"}}
scale_trend : {"const": {"L3": 0.1890688202878888, "L4": 0.2532447175918854, "L6_pred": 0.4381042996650741}, "scaled": {"L3": 0.2661542799189741, "L4": 0.41400211816737426, "L6_pred": 0.9480902669410374}}
L6 : {"const": {"best": 1.0210189581611806, "at_pred": 1.0224147951068283, "

{'state': {'status': 'stopped',
  'pool_asset': 'azureml:wdladder-pool:1',
  'gens': [{'gen': 1,
    'job': 'lucid_store_2f8cns63cp',
    'start_progress': 0,
    'start': 'Thu Oct 22 04:30:46 2026'},
   {'gen': 2,
    'job': 'affable_dog_f2fc1fvrzr',
    'start_progress': 60000000,
    'start': 'Thu Oct 22 08:21:45 2026'},
   {'gen': 3,
    'job': 'affable_dog_f2fc1fvrzr',
    'start_progress': 110000000,
    'start': 'Thu Oct 22 11:43:35 2026'},
   {'gen': 4,
    'job': 'kind_rose_x0s00217lq',
    'start_progress': 360000000,
    'start': 'Fri Oct 23 03:43:01 2026'},
   {'gen': 5,
    'job': 'jovial_seal_h8c6k46g8q',
    'start_progress': 420000000,
    'start': 'Sat Oct 24 03:50:26 2026'},
   {'gen': 6,
    'job': 'funny_hand_1f4m1b9szq',
    'start_progress': 450000000,
    'start': 'Sat Oct 24 20:45:27 2026'},
   {'gen': 7,
    'job': 'funny_hand_1f4m1b9szq',
    'start_progress': 580003107,
    'start': 'Sun Oct 25 04:56:08 2026'},
   {'gen': 8,
    'job': 'funny_hand_1f4m1b9szq'

In [4]:
import glob, json, re, numpy as np
R = "/home/azureuser/wdladder_data/runs"
pat = re.compile(r"^L(\d+)_t120_cos_(const|scaled)_lr([\d.]+)_p([\d.]+)_s(\d+)$")
D = {}
for d in glob.glob(R + "/L*_t120_cos_*"):
    m = pat.match(os.path.basename(d))
    if not m or float(m[3]) != 1.0: continue
    try: s = json.load(open(d + "/summary.json"))
    except Exception: continue
    if s.get("status") == "done":
        D.setdefault((int(m[1]), m[2]), {}).setdefault(float(m[4]), []).append(s["final"]["bpb"])
def fit(pts):
    ps = sorted(pts); x = np.log2(ps); y = np.array([np.mean(pts[p]) for p in ps])
    if len(ps) >= 3:
        k = np.polyfit(x, y, 2); a, b = k[0], k[1]
        xm = float(np.clip(-b/(2*a), x.min()-1, x.max()+1)) if a > 0 else float(x[y.argmin()])
        ymin = min(float(np.polyval(k, xm)), float(y.min()))
        y1 = float(np.mean(pts[1.0])) if 1.0 in pts else float(np.polyval(k, 0.0))
    else:
        xm, ymin = float(x[y.argmin()]), float(y.min())
        y1 = float(np.mean(pts[1.0])) if 1.0 in pts else float(np.interp(0.0, x, y))
    return dict(n=len(ps), p_opt=round(2**xm, 3), bpb_opt=ymin, bpb_p1=y1)
print(f"{'L':>3} {'untuned gap':>12} {'tuned gap':>10} {'survival':>9} {'p_opt c/s':>16} {'WD ratio':>9}")
for L in sorted({k[0] for k in D}):
    if (L, "const") not in D or (L, "scaled") not in D: continue
    c, s = fit(D[(L, "const")]), fit(D[(L, "scaled")])
    ug, tg = c["bpb_p1"] - s["bpb_p1"], c["bpb_opt"] - s["bpb_opt"]
    print(f"{L:>3} {ug:12.5f} {tg:10.5f} {tg/ug if ug > 0 else float('nan'):9.3f} {c['p_opt']:>7}/{s['p_opt']:<7} {s['p_opt']/c['p_opt']:9.2f}")
for k in sorted(D): print(k, {p: round(float(np.mean(v)), 5) for p, v in sorted(D[k].items())})

  L  untuned gap  tuned gap  survival        p_opt c/s  WD ratio
  3      0.03499    0.00360     0.103   0.192/0.269        1.40
  4      0.03105    0.00402     0.130   0.253/0.414        1.64
  6      0.02407    0.00513     0.213   0.331/0.632        1.91
(3, 'const') {0.125: 1.27778, 0.25: 1.27925, 0.5: 1.29115, 1.0: 1.32209, 2.0: 1.37421, 4.0: 1.43913}
(3, 'scaled') {0.125: 1.27648, 0.25: 1.27474, 0.5: 1.27589, 1.0: 1.2871, 2.0: 1.31112, 4.0: 1.34722, 8.0: 1.38583}
(4, 'const') {0.25: 1.18881, 0.5: 1.19805, 1.0: 1.22397, 2.0: 1.26833, 4.0: 1.331}
(4, 'scaled') {0.5: 1.18576, 1.0: 1.19292, 2.0: 1.21365, 4.0: 1.24669, 8.0: 1.28723}
(6, 'const') {0.354: 1.02102, 0.707: 1.03145, 1.41: 1.05772, 2.83: 1.1022}
(6, 'scaled') {0.708: 1.01592, 1.41: 1.02416, 2.83: 1.0431, 5.66: 1.07497}


In [2]:
import glob, json
R = "/home/azureuser/wdladder_data/runs"
AJ.sync()
L6 = {}
for d in sorted(glob.glob(R + "/L6_*")):
    n = d.split("/")[-1]
    try: s = json.load(open(d + "/summary.json")); L6[n] = s["final"]["bpb"] if s.get("status") == "done" else None
    except Exception: L6[n] = None
    p = chain._remote_state(n); print(f"{n:45s} {p:15s} bpb={L6[n]}")
def get(meth, untuned):
    return next((v for k, v in L6.items() if f"_{meth}_" in k and (("_lr1_p1_" in k) == untuned)), None)
tc, ts, uc, us = get("const", False), get("scaled", False), get("const", True), get("scaled", True)
if tc and ts: print(f"L6 tuned gap   = {tc-ts:.5f}   (L3 tuned gap 0.00261)")
if uc and us: print(f"L6 untuned gap = {uc-us:.5f}   (L3 untuned gap 0.03499)")
if tc and ts and uc and us: print(f"L6 survival    = {(tc-ts)/(uc-us):.3f}   (L3 0.074)")
if not ((tc and ts) or (uc and us)): print("no complete pair yet -> wait")

synced 425 files -> /home/azureuser/wdladder_data/runs
L6_t120_cos_const_lr1_p0.354_s0               done            bpb=1.0210189581611806
L6_t120_cos_const_lr1_p0.707_s0               done            bpb=1.0314510663357572
L6_t120_cos_const_lr1_p1.41_s0                done            bpb=1.0577158685395203
L6_t120_cos_const_lr1_p2.83_s0                done            bpb=1.1021956063122296
L6_t120_cos_scaled_lr1_p0.708_s0              done            bpb=1.0159213188915845
L6_t120_cos_scaled_lr1_p1.41_s0               partial@24999   bpb=None
L6 tuned gap   = 0.00510   (L3 tuned gap 0.00261)


In [2]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 3

synced 434 files -> /home/azureuser/wdladder_data/runs
79 runs
runs loaded: 79 (78 done)
GATE 1: PASS {'gain_mean': 0.03499458361150165, 'threshold': 0.003, 'speedup_untuned': 1.5183786483006478}
edge_extension : []
wd_ratio_cos : 1.4080200039912556
tuned_gap : {"gap": 0.002583278761996155, "ci": [0.0010553279314982537, 0.0041112295924940565], "const": [1.2786254515921505, 1.278447403896516], "scaled": [1.2756821067972361, 1.276224191167438]}
survival_fraction : 0.07381938847093783
hypothesis_verdict : {"cos": {"measured": 1.4080200039912556, "ci": [0.877382105149401, 2.254021055010117], "consistent": ["H_none", "H_int"], "closest": "H_int"}, "wsd": {"measured": 0.28702679510801843, "ci": [0.19135119673867895, 0.43054019266202764], "consistent": [], "closest": "H_none"}}
OUTCOME: O1 genuine: Muon-SW keeps 7% of its gain after both methods are tuned
report -> /home/azureuser/wdladder_data/report/report.md


In [4]:
import glob, json, statistics as S
R = "/home/azureuser/wdladder_data/runs"
s0 = json.load(open(sorted(glob.glob(R + "/L3_t120_cos_*_s1/summary.json"))[0]))
print("summary keys:", list(s0))
def num(v):
    return v if isinstance(v, (int, float)) else (v.get("all", v.get("total", v.get("mean"))) if isinstance(v, dict) else None)
KEY = next((k for k in ("bpb", "final_bpb", "eval_bpb", "bpb_all", "final_eval_bpb", "val_bpb") if num(s0.get(k)) is not None),
           next((k for k, v in s0.items() if "bpb" in k.lower() and num(v) is not None), None))
print("using key:", KEY)
def bpb(n):
    try: return num(json.load(open(f"{R}/{n}/summary.json")).get(KEY))
    except Exception: return None
T = {}
for d in sorted(glob.glob(R + "/L3_t120_cos_*_s[1-4]")):
    n = d.split("/")[-1]; T.setdefault(n.rsplit("_s", 1)[0], {})[int(n.rsplit("_s", 1)[1])] = bpb(n)
T = {k: v for k, v in T.items() if len(v) >= 3}
for k, v in T.items(): print(k, v)
c = next((v for k, v in T.items() if "_const_" in k), {}); s = next((v for k, v in T.items() if "_scaled_" in k), {})
d = [c[i] - s[i] for i in sorted(set(c) & set(s)) if c[i] is not None and s[i] is not None]
TC = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776, 6: 2.571}
if len(d) >= 2:
    m = S.mean(d); se = S.stdev(d) / len(d) ** 0.5; t = TC.get(len(d), 2.0)
    print(f"paired gap {m:.5f}  95% CI [{m-t*se:.5f}, {m+t*se:.5f}]  n={len(d)}  per-seed {[round(x,5) for x in d]}")
    print("survival vs untuned 0.035:", round(m / 0.03499458, 3), "| CI above 0.002 null:", m - t*se > 0.002)
else:
    print("not enough paired values:", d)
!{PY} analysis.py --stage 4

summary keys: ['status', 'final', 'step', 'init_loss', 'tokens', 'flops', 'wall_seconds', 'segments', 'device', 'tok_per_s', 'spec']
using key: None
L3_t120_cos_const_lr0.924_p0.186 {1: None, 2: None, 3: None, 4: None}
L3_t120_cos_scaled_lr0.917_p0.26 {1: None, 2: None, 3: None, 4: None}
not enough paired values: []
runs loaded: 79 (78 done)
GATE 1: PASS {'gain_mean': 0.03499458361150165, 'threshold': 0.003, 'speedup_untuned': 1.5183786483006478}
edge_extension : []
wd_ratio_cos : 1.4080200039912556
tuned_gap : {"gap": 0.002583278761996155, "ci": [0.0010553279314982537, 0.0041112295924940565], "const": [1.2786254515921505, 1.278447403896516], "scaled": [1.2756821067972361, 1.276224191167438]}
survival_fraction : 0.07381938847093783
hypothesis_verdict : {"cos": {"measured": 1.4080200039912556, "ci": [0.877382105149401, 2.254021055010117], "consistent": ["H_none", "H_int"], "closest": "H_int"}, "wsd": {"measured": 0.28702679510801843, "ci": [0.19135119673867895, 0.43054019266202764], "co

In [5]:
import glob, json, statistics as S
R = "/home/azureuser/wdladder_data/runs"
s0 = json.load(open(sorted(glob.glob(R + "/L3_t120_cos_*_s1/summary.json"))[0]))
print("final =", json.dumps(s0["final"])[:600])
def flat(v, p=""):
    if isinstance(v, dict):
        for k, x in v.items(): yield from flat(x, f"{p}.{k}" if p else k)
    elif isinstance(v, (int, float)): yield p, v
F = dict(flat(s0["final"]))
KEY = next((k for k in ("bpb", "all", "bpb.all", "total", "bpb.total", "mean", "bpb.mean") if k in F),
           next((k for k in F if "bpb" in k.lower()), None))
print("using final key:", KEY)
def bpb(n):
    try: return dict(flat(json.load(open(f"{R}/{n}/summary.json"))["final"])).get(KEY)
    except Exception: return None
T = {}
for d in sorted(glob.glob(R + "/L3_t120_cos_*_s[1-4]")):
    n = d.split("/")[-1]; T.setdefault(n.rsplit("_s", 1)[0], {})[int(n.rsplit("_s", 1)[1])] = bpb(n)
T = {k: v for k, v in T.items() if len(v) >= 3}
for k, v in T.items(): print(k, v)
c = next((v for k, v in T.items() if "_const_" in k), {}); s = next((v for k, v in T.items() if "_scaled_" in k), {})
d = [c[i] - s[i] for i in sorted(set(c) & set(s)) if c[i] is not None and s[i] is not None]
TC = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776}
if len(d) >= 2:
    m = S.mean(d); se = S.stdev(d) / len(d) ** 0.5; t = TC.get(len(d), 2.0)
    print(f"paired gap {m:.5f}  95% CI [{m-t*se:.5f}, {m+t*se:.5f}]  n={len(d)}  per-seed {[round(x,5) for x in d]}")
    print("survival vs untuned 0.035:", round(m / 0.03499458, 3), "| CI lower bound > 0.002 null:", m - t*se > 0.002)
else:
    print("not enough paired values:", d)

final = {"web": {"bpb": 1.3307954513701, "loss": 3.5298768761323136}, "code": {"bpb": 1.0081488091027753, "loss": 2.220716905204479}, "math": {"bpb": 1.5195528719917486, "loss": 3.3123311529740933}, "bpb": 1.278447403896516, "bpb_macro": 1.2861657108215412, "step": 6930, "tokens": 227082240}
using final key: bpb
L3_t120_cos_const_lr0.924_p0.186 {1: 1.278447403896516, 2: 1.2786254515921505, 3: 1.2767037628073559, 4: 1.2789081767014145}
L3_t120_cos_scaled_lr0.917_p0.26 {1: 1.2756821067972361, 2: 1.276224191167438, 3: 1.2746442232995574, 4: 1.2757100035143671}
paired gap 0.00261  95% CI [0.00183, 0.00338]  n=4  per-seed [0.00277, 0.0024, 0.00206, 0.0032]
survival vs untuned 0.035: 0.074 | CI lower bound > 0.002 null: False


In [2]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 3
!{PY} llm_review.py review --stage 3

synced 399 files -> /home/azureuser/wdladder_data/runs
69 runs
runs loaded: 69 (69 done)
GATE 1: PASS {'gain_mean': 0.03499458361150165, 'threshold': 0.003, 'speedup_untuned': 1.5183786483006478}
edge_extension : []
wd_ratio_cos : 1.4080200039912556
tuned_gap : {"gap": 0.002583278761996155, "ci": [0.0015270611895128144, 0.0036394963344794956], "const": [1.2786254515921505, 1.278447403896516], "scaled": [1.2756821067972361, 1.276224191167438]}
survival_fraction : 0.07381938847093783
hypothesis_verdict : {"cos": {"measured": 1.4080200039912556, "ci": [0.877382105149401, 2.254021055010117], "consistent": ["H_none", "H_int"], "closest": "H_int"}, "wsd": {"measured": 0.28702679510801843, "ci": [0.19135119673867895, 0.43054019266202764], "consistent": [], "closest": "H_none"}}
OUTCOME: O1 genuine: Muon-SW keeps 7% of its gain after both methods are tuned
report -> /home/azureuser/wdladder_data/report/report.md
### 1. Anomalies and checks
- **Noise versus effect:** Gate 1’s gain, `0.034994583

In [3]:
import plan, json
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
T = [r for r in AJ.stage_runs([3], 120) if r["sched"] == "cos" and r["seed"] in (1, 2)]
EXTRA = [dict(plan.spec(3, r["L"], r["tpp"], "cos", r["method"], r["lr_mult"], r["prod_mult"], s), device="gpu")
         for r in T if r["seed"] == 1 for s in (3, 4)]
R5 = AJ.stage_runs([5], 120); s0 = min(r["seed"] for r in R5)
K5 = [dict(r, device="gpu") for r in R5 if r["seed"] == s0]
print("extra seeds:", [r["name"] for r in EXTRA])
print("L6:", len(R5), "->", len(K5), [r["name"] for r in K5])
chain.add(EXTRA + K5, cid="gpu")
R = "/home/azureuser/wdladder_data/runs"
for r in sorted([r for r in AJ.stage_runs([3], 120) if r["sched"] == "wsd"], key=lambda r: (r["method"], r["prod_mult"])):
    s = json.load(open(f"{R}/{r['name']}/summary.json"))
    print(f"{r['method']:7s} prod={r['prod_mult']:<6} bpb={s.get('bpb', s.get('final_bpb'))}")

extra seeds: ['L3_t120_cos_const_lr0.924_p0.186_s3', 'L3_t120_cos_const_lr0.924_p0.186_s4', 'L3_t120_cos_scaled_lr0.917_p0.26_s3', 'L3_t120_cos_scaled_lr0.917_p0.26_s4']
L6: 8 -> 8 ['L6_t120_cos_const_lr1_p0.354_s0', 'L6_t120_cos_const_lr1_p0.707_s0', 'L6_t120_cos_const_lr1_p1.41_s0', 'L6_t120_cos_const_lr1_p2.83_s0', 'L6_t120_cos_scaled_lr1_p0.708_s0', 'L6_t120_cos_scaled_lr1_p1.41_s0', 'L6_t120_cos_scaled_lr1_p2.83_s0', 'L6_t120_cos_scaled_lr1_p5.66_s0']
[chain gpu] +12 runs (queue 81)
[chain gpu] started worker job elated_collar_2bk6pzvnz2
const   prod=0.25   bpb=None
const   prod=0.5    bpb=None
const   prod=1.0    bpb=None
const   prod=2.0    bpb=None
const   prod=4.0    bpb=None
scaled  prod=0.5    bpb=None
scaled  prod=1.0    bpb=None
scaled  prod=2.0    bpb=None
scaled  prod=4.0    bpb=None
scaled  prod=8.0    bpb=None


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading wdladder (0.27 MBs): 100%|██

In [2]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 2
!{PY} llm_review.py review --stage 2

synced 258 files -> /home/azureuser/wdladder_data/runs
45 runs
/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/analysis.py:427: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=6)
runs loaded: 45 (45 done)
GATE 1: PASS {'gain_mean': 0.03499458361150165, 'threshold': 0.003, 'speedup_untuned': 1.5183786483006478}
edge_extension : []
wd_ratio_cos : 1.4080200039912556
OUTCOME: O0 untuned gain confirmed (gate 1 PASS); tuned comparison pending
report -> /home/azureuser/wdladder_data/report/report.md
### 1. Anomalies and missing diagnostics
- **No obvious noise-dominated effect:** `gain_mean = 0.03499458361150165` comfortably exceeds `seed_noise = 0.0010182919981824856`; both listed gains are positive and closely agree. However, `seed_sd = 0.0007066882225594238` differs from `seed_noise`; their definitions need 

In [3]:
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
print({s: len(AJ.stage_runs([s], 120)) for s in (3, 4)})
GO([3, 4])

{3: 14, 4: 10}
[chain gpu] +24 runs (queue 69)
[chain gpu] started worker job funny_hand_1f4m1b9szq
{'waiting': 24}


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading wdladder (0.26 MBs): 100%|██

{'L3_t120_cos_const_lr0.924_p0.186_s1': 'waiting',
 'L3_t120_cos_const_lr0.924_p0.186_s2': 'waiting',
 'L3_t120_wsd_const_lr1_p0.25_s0': 'waiting',
 'L3_t120_wsd_const_lr1_p0.5_s0': 'waiting',
 'L3_t120_wsd_const_lr1_p1_s0': 'waiting',
 'L3_t120_wsd_const_lr1_p2_s0': 'waiting',
 'L3_t120_wsd_const_lr1_p4_s0': 'waiting',
 'L3_t120_cos_scaled_lr0.917_p0.26_s1': 'waiting',
 'L3_t120_cos_scaled_lr0.917_p0.26_s2': 'waiting',
 'L3_t120_wsd_scaled_lr1_p0.5_s0': 'waiting',
 'L3_t120_wsd_scaled_lr1_p1_s0': 'waiting',
 'L3_t120_wsd_scaled_lr1_p2_s0': 'waiting',
 'L3_t120_wsd_scaled_lr1_p4_s0': 'waiting',
 'L3_t120_wsd_scaled_lr1_p8_s0': 'waiting',
 'L4_t120_cos_const_lr1_p0.25_s0': 'waiting',
 'L4_t120_cos_const_lr1_p0.5_s0': 'waiting',
 'L4_t120_cos_const_lr1_p1_s0': 'waiting',
 'L4_t120_cos_const_lr1_p2_s0': 'waiting',
 'L4_t120_cos_const_lr1_p4_s0': 'waiting',
 'L4_t120_cos_scaled_lr1_p0.5_s0': 'waiting',
 'L4_t120_cos_scaled_lr1_p1_s0': 'waiting',
 'L4_t120_cos_scaled_lr1_p2_s0': 'waiting',


In [2]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 2
!{PY} llm_review.py review --stage 2

synced 246 files -> /home/azureuser/wdladder_data/runs
42 runs
/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/analysis.py:427: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=6)
runs loaded: 42 (42 done)
GATE 1: PASS {'gain_mean': 0.03499458361150165, 'threshold': 0.003, 'speedup_untuned': 1.5183786483006478}
edge_extension : [{"method": "scaled", "lr_mult": 0.5, "prod_mult": 0.125}, {"method": "scaled", "lr_mult": 1.0, "prod_mult": 0.125}, {"method": "scaled", "lr_mult": 2.0, "prod_mult": 0.125}]
wd_ratio_cos : 1.6850477035071827
OUTCOME: O0 untuned gain confirmed (gate 1 PASS); tuned comparison pending
report -> /home/azureuser/wdladder_data/report/report.md
### 1. Anomalies and checks
- **Seed noise versus effect:** No obvious anomaly. Gate-1 gains are nearly identical (`0.03501365`, `0.03497552`) an

In [3]:
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
print(len(AJ.stage_runs([2], 120)), "stage-2 runs")
GO([2])

33 stage-2 runs
[chain gpu] +3 runs (queue 45)
[chain gpu] started worker job jovial_seal_h8c6k46g8q
{'done': 30, 'waiting': 3}


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading wdladder (0.25 MBs): 100%|██

{'L3_t120_cos_const_lr0.5_p0.25_s0': 'done',
 'L3_t120_cos_const_lr0.5_p0.5_s0': 'done',
 'L3_t120_cos_const_lr0.5_p1_s0': 'done',
 'L3_t120_cos_const_lr0.5_p2_s0': 'done',
 'L3_t120_cos_const_lr0.5_p4_s0': 'done',
 'L3_t120_cos_const_lr1_p0.25_s0': 'done',
 'L3_t120_cos_const_lr1_p0.5_s0': 'done',
 'L3_t120_cos_const_lr1_p2_s0': 'done',
 'L3_t120_cos_const_lr1_p4_s0': 'done',
 'L3_t120_cos_const_lr2_p0.25_s0': 'done',
 'L3_t120_cos_const_lr2_p0.5_s0': 'done',
 'L3_t120_cos_const_lr2_p1_s0': 'done',
 'L3_t120_cos_const_lr2_p2_s0': 'done',
 'L3_t120_cos_const_lr2_p4_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p0.5_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p1_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p2_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p4_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p8_s0': 'done',
 'L3_t120_cos_scaled_lr1_p0.5_s0': 'done',
 'L3_t120_cos_scaled_lr1_p2_s0': 'done',
 'L3_t120_cos_scaled_lr1_p4_s0': 'done',
 'L3_t120_cos_scaled_lr1_p8_s0': 'done',
 'L3_t120_cos_scaled_lr2_p0.5_s0

In [4]:
chain.status("gpu", runs=False); DONE([2])

[chain gpu] status=finished owner=None heartbeat=2570 s ago successor=clever_coconut_01pb5024jc
   gen {'gen': 1, 'job': 'lucid_store_2f8cns63cp', 'start_progress': 0, 'start': 'Thu Oct 22 04:30:46 2026'}
   gen {'gen': 2, 'job': 'affable_dog_f2fc1fvrzr', 'start_progress': 60000000, 'start': 'Thu Oct 22 08:21:45 2026'}
   gen {'gen': 3, 'job': 'affable_dog_f2fc1fvrzr', 'start_progress': 110000000, 'start': 'Thu Oct 22 11:43:35 2026'}
   active jobs: none
{'done': 30, 'waiting': 6}


{'L3_t120_cos_const_lr0.5_p0.25_s0': 'done',
 'L3_t120_cos_const_lr0.5_p0.5_s0': 'done',
 'L3_t120_cos_const_lr0.5_p1_s0': 'done',
 'L3_t120_cos_const_lr0.5_p2_s0': 'done',
 'L3_t120_cos_const_lr0.5_p4_s0': 'done',
 'L3_t120_cos_const_lr1_p0.25_s0': 'done',
 'L3_t120_cos_const_lr1_p0.5_s0': 'done',
 'L3_t120_cos_const_lr1_p2_s0': 'done',
 'L3_t120_cos_const_lr1_p4_s0': 'done',
 'L3_t120_cos_const_lr2_p0.25_s0': 'done',
 'L3_t120_cos_const_lr2_p0.5_s0': 'done',
 'L3_t120_cos_const_lr2_p1_s0': 'done',
 'L3_t120_cos_const_lr2_p2_s0': 'done',
 'L3_t120_cos_const_lr2_p4_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p0.5_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p1_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p2_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p4_s0': 'done',
 'L3_t120_cos_scaled_lr0.5_p8_s0': 'done',
 'L3_t120_cos_scaled_lr1_p0.5_s0': 'done',
 'L3_t120_cos_scaled_lr1_p2_s0': 'done',
 'L3_t120_cos_scaled_lr1_p4_s0': 'done',
 'L3_t120_cos_scaled_lr1_p8_s0': 'done',
 'L3_t120_cos_scaled_lr2_p0.5_s0

In [3]:
import shutil, glob
def CLEAN():
    AJ.sync()
    R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
    os.makedirs(X, exist_ok=True)
    for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
        shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
    print(len([d for d in os.listdir(R) if not d.startswith("_")]), "runs")
CLEAN()
!{PY} analysis.py --stage 2
!{PY} llm_review.py review --stage 2

synced 240 files -> /home/azureuser/wdladder_data/runs
36 runs
/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/analysis.py:427: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=6)
runs loaded: 36 (36 done)
GATE 1: PASS {'gain_mean': 0.03499458361150165, 'threshold': 0.003, 'speedup_untuned': 1.5183786483006478}
edge_extension : [{"method": "const", "lr_mult": 0.5, "prod_mult": 0.125}, {"method": "const", "lr_mult": 1.0, "prod_mult": 0.125}, {"method": "const", "lr_mult": 2.0, "prod_mult": 0.125}, {"method": "scaled", "lr_mult": 0.5, "prod_mult": 0.25}, {"method": "scaled", "lr_mult": 1.0, "prod_mult": 0.25}, {"method": "scaled", "lr_mult": 2.0, "prod_mult": 0.25}]
wd_ratio_cos : 1.5284375486180128
OUTCOME: O0 untuned gain confirmed (gate 1 PASS); tuned comparison pending
report -> /home/azureuser/wdladder

In [2]:
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
chain.status("gpu"); DONE([1])

[chain gpu] status=finished owner=None heartbeat=2965 s ago successor=frank_bucket_ptp34xpg8g
   gen {'gen': 1, 'job': 'lucid_store_2f8cns63cp', 'start_progress': 0, 'start': 'Thu Oct 22 04:30:46 2026'}
   active jobs: none
   L3_t120_cos_const_lr1_p1_s0                   done
   L3_t120_cos_const_lr1_p1_s1                   done
   L3_t120_cos_scaled_lr1_p1_s0                  done
   L3_t120_cos_scaled_lr1_p1_s1                  done
   L3_t30_cos_const_lr1_p1_s0                    done
   L3_t30_cos_scaled_lr1_p1_s0                   done
{'done': 6}


{'L3_t120_cos_const_lr1_p1_s0': 'done',
 'L3_t120_cos_const_lr1_p1_s1': 'done',
 'L3_t120_cos_scaled_lr1_p1_s0': 'done',
 'L3_t120_cos_scaled_lr1_p1_s1': 'done',
 'L3_t30_cos_const_lr1_p1_s0': 'done',
 'L3_t30_cos_scaled_lr1_p1_s0': 'done'}

In [4]:
import shutil, glob
AJ.sync()
R = "/home/azureuser/wdladder_data/runs"; X = "/home/azureuser/wdladder_data/runs_excluded"
os.makedirs(X, exist_ok=True)
for d in glob.glob(R + "/*smoke*") + glob.glob(R + "/bench_*"):
    shutil.rmtree(os.path.join(X, os.path.basename(d)), ignore_errors=True); shutil.move(d, X)
print(sorted(os.listdir(R)))
!{PY} analysis.py --stage 1
!{PY} llm_review.py review --stage 1

synced 56 files -> /home/azureuser/wdladder_data/runs
['L3_t120_cos_const_lr1_p1_s0', 'L3_t120_cos_const_lr1_p1_s1', 'L3_t120_cos_scaled_lr1_p1_s0', 'L3_t120_cos_scaled_lr1_p1_s1', 'L3_t30_cos_const_lr1_p1_s0', 'L3_t30_cos_scaled_lr1_p1_s0']
/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/analysis.py:427: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=6)
runs loaded: 6 (6 done)
GATE 1: PASS {'gain_mean': 0.03499458361150165, 'threshold': 0.003, 'speedup_untuned': 1.4425162600178956}
OUTCOME: O0 untuned gain confirmed (gate 1 PASS); tuned comparison pending
report -> /home/azureuser/wdladder_data/report/report.md
### 1. Anomalies and missing diagnostics

- **Seed noise versus effect:** No apparent anomaly. Mean gain is **0.03499458361150165**, versus reported noise **0.0010182919981824856**; both seed ga

In [5]:
def GO(stages, l6=120):
    chain.add([dict(r, device="gpu") for r in AJ.stage_runs(stages, l6)], cid="gpu")
    return DONE(stages, l6)
GO([2])

[chain gpu] +30 runs (queue 36)
[chain gpu] started worker job affable_dog_f2fc1fvrzr
{'waiting': 30}


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading wdladder (0.24 MBs): 100%|██

{'L3_t120_cos_const_lr0.5_p0.25_s0': 'waiting',
 'L3_t120_cos_const_lr0.5_p0.5_s0': 'waiting',
 'L3_t120_cos_const_lr0.5_p1_s0': 'waiting',
 'L3_t120_cos_const_lr0.5_p2_s0': 'waiting',
 'L3_t120_cos_const_lr0.5_p4_s0': 'waiting',
 'L3_t120_cos_const_lr1_p0.25_s0': 'waiting',
 'L3_t120_cos_const_lr1_p0.5_s0': 'waiting',
 'L3_t120_cos_const_lr1_p2_s0': 'waiting',
 'L3_t120_cos_const_lr1_p4_s0': 'waiting',
 'L3_t120_cos_const_lr2_p0.25_s0': 'waiting',
 'L3_t120_cos_const_lr2_p0.5_s0': 'waiting',
 'L3_t120_cos_const_lr2_p1_s0': 'waiting',
 'L3_t120_cos_const_lr2_p2_s0': 'waiting',
 'L3_t120_cos_const_lr2_p4_s0': 'waiting',
 'L3_t120_cos_scaled_lr0.5_p0.5_s0': 'waiting',
 'L3_t120_cos_scaled_lr0.5_p1_s0': 'waiting',
 'L3_t120_cos_scaled_lr0.5_p2_s0': 'waiting',
 'L3_t120_cos_scaled_lr0.5_p4_s0': 'waiting',
 'L3_t120_cos_scaled_lr0.5_p8_s0': 'waiting',
 'L3_t120_cos_scaled_lr1_p0.5_s0': 'waiting',
 'L3_t120_cos_scaled_lr1_p2_s0': 'waiting',
 'L3_t120_cos_scaled_lr1_p4_s0': 'waiting',
 'L3_t1

In [3]:
AJ.sync()
!{PY} analysis.py --stage 1
!{PY} llm_review.py review --stage 1

synced 62 files -> /home/azureuser/wdladder_data/runs
/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/analysis.py:427: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(fontsize=6)
runs loaded: 10 (10 done)
GATE 1: FAIL {'gain_mean': -0.7282254403899429, 'threshold': 3.054840416777544, 'speedup_untuned': 0.011950910676508622}
OUTCOME: O3 non-replication: Muon-SW shows no gain over const decay at MAI hparams on the MoE ladder
report -> /home/azureuser/wdladder_data/report/report.md
### 1. Anomalies and missing diagnostics
- **Noise dominates:** mean gain is **−0.728225**, versus reported seed noise **1.527420** and threshold **3.054840**. Paired gains disagree sharply: **−1.491426** and **+0.034976**.
- **Severely degraded run:** scaled loss **2.814026** versus paired constant loss **1.322600** drives the negative m

In [6]:
import time
ml = AJ.client(); J = ["silly_oregano_xy0bmjk3rw", "epic_bucket_cffmcd7sn1", "crimson_jelly_4gdpdmmrbq"]
for j in J: ml.jobs.begin_cancel(j)
time.sleep(90); print({j: ml.jobs.get(j).status for j in J})
cont = AJ.container()
def CLEAR(runs):
    for r in runs:
        bl = [b.name for b in cont.list_blobs(name_starts_with=f"{C.AZURE['remote_prefix']}/runs/{r['name']}/")]
        for b in bl: cont.delete_blob(b)
        print(r["name"], "cleared", len(bl))
R1 = AJ.stage_runs([1], 120); CLEAR(R1)
chain.add([dict(r, device="gpu") for r in R1], cid="gpu")

{'silly_oregano_xy0bmjk3rw': 'Canceled', 'epic_bucket_cffmcd7sn1': 'Canceled', 'crimson_jelly_4gdpdmmrbq': 'Canceled'}
L3_t120_cos_const_lr1_p1_s0 cleared 4
L3_t120_cos_const_lr1_p1_s1 cleared 4
L3_t120_cos_scaled_lr1_p1_s0 cleared 5
L3_t120_cos_scaled_lr1_p1_s1 cleared 5
L3_t30_cos_const_lr1_p1_s0 cleared 0
L3_t30_cos_scaled_lr1_p1_s0 cleared 0
[chain gpu] +6 runs (queue 6)
[chain gpu] started worker job lucid_store_2f8cns63cp


Uploading wdladder (0.22 MBs): 100%|██████████| 224369/224369 [00:00<00:00, 5658097.76it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


'lucid_store_2f8cns63cp'

In [2]:
chain.status("bench", runs=False)
for n in [f"bench_cpu_L{L}" for L in (3, 4)] + [f"bench_gpu_L{L}" for L in (3, 4, 6, 9)]:
    print(n, AJ.read_remote_json(n, "bench.json"))

[chain bench] status=finished owner=None heartbeat=1479 s ago successor=polite_sprout_5ckzqvdpgr
   gen {'gen': 1, 'job': 'sleepy_rod_n64t8m6fyr', 'start_progress': 0, 'start': 'Thu Oct 22 03:31:51 2026'}
   active jobs: none
bench_cpu_L3 {'L': 3, 'device': 'cpu', 'threads': 8, 'micro': 16, 'tok_per_s': 4173.588136473241, 'peak_mem_gb': None, 'active': 1892352, 'total': 14934016, 'time': 'Thu Oct 22 03:19:44 2026'}
bench_cpu_L4 {'L': 4, 'device': 'cpu', 'threads': 8, 'micro': 16, 'tok_per_s': 2846.7476110771304, 'peak_mem_gb': None, 'active': 3891200, 'total': 27566080, 'time': 'Thu Oct 22 03:20:00 2026'}
bench_gpu_L3 {'L': 3, 'device': 'cuda', 'threads': 4, 'micro': 64, 'tok_per_s': 122130.9393970361, 'peak_mem_gb': 5.51564032, 'active': 1892352, 'total': 14934016, 'time': 'Thu Oct 22 03:32:25 2026'}
bench_gpu_L4 {'L': 4, 'device': 'cuda', 'threads': 4, 'micro': 64, 'tok_per_s': 73642.18033309087, 'peak_mem_gb': 6.989180928, 'active': 3891200, 'total': 27566080, 'time': 'Thu Oct 22 03

In [3]:
AJ.sync()
!{PY} plan.py --budget

synced 30 files -> /home/azureuser/wdladder_data/runs
throughput (tok/s): {'gpu-L9': 4372, 'gpu-L3': 122131, 'gpu-L6': 21583, 'gpu-L4': 73642, 'cpu-L3': 4174, 'cpu-L4': 2847} (measured)
pool requirement (M unique tokens): {'web': 1085, 'code': 113, 'math': 68}

stage                  runs run-hours  $ stage   $ cum measured
1 L3 gate                 6      72.1    41.78   41.78 True
2 L3 grid                30     488.6   283.21  324.99 True
3 L3 confirm+WSD         14     224.3   130.00  454.99 True
4 L4 GPU                 10      18.7    11.29  466.28 True
5 L6 GPU tpp60            8      98.5    59.58  525.86 True
6 L9 GPU (opt)            2     204.7   123.80  649.66 True
budget $140; add compute-instance hours x $1.31/h (stop it when idle)


In [4]:
chain.checkin([1], l6_tpp=120)

synced 26 files -> /home/azureuser/wdladder_data/runs
[budget] spent $0.02 + in-flight $0.00 + this batch $41.78 = $41.80 / $140
[submit] silly_oregano_xy0bmjk3rw cpu ['L3_t120_cos_const_lr1_p1_s0', 'L3_t120_cos_const_lr1_p1_s1']
[submit] epic_bucket_cffmcd7sn1 cpu ['L3_t120_cos_scaled_lr1_p1_s0', 'L3_t120_cos_scaled_lr1_p1_s1']
[submit] crimson_jelly_4gdpdmmrbq cpu ['L3_t30_cos_const_lr1_p1_s0', 'L3_t30_cos_scaled_lr1_p1_s0']
[chain gpu] status=None owner=None heartbeat=- successor=None
   active jobs: none


Uploading wdladder (0.22 MBs): 100%|██████████| 220432/220432 [00:00<00:00, 4575189.25it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
Uploading wdladder (0.22 MBs): 100%|██████████| 222524/222524 [00:00<00:00, 5668279.92it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


In [5]:
import plan
mk = lambda dev, L: dict(plan.spec(0, L, 120, "cos", "const", 1, 1, 0), name=f"bench_{dev}_L{L}", device=dev)
print([(r["name"], r["L"]) for r in [mk("cpu", 3), mk("gpu", 9)]])
print(AJ.submit([mk("cpu", L) for L in (3, 4)], pool_asset="azureml:wdladder-pool:1",
                extra_args="--benchmark 20", usd_override=0.08, tag_stage="bench"))
chain.add([mk("gpu", L) for L in (3, 4, 6, 9)], cid="bench", extra="--benchmark 20")

[('bench_cpu_L3', 3), ('bench_gpu_L9', 9)]
[budget] spent $0.00 + in-flight $0.00 + this batch $0.16 = $0.16 / $140
[submit] nifty_pocket_9nw8gldfsd cpu ['bench_cpu_L3', 'bench_cpu_L4']
['nifty_pocket_9nw8gldfsd']
[chain bench] +4 runs (queue 4)
[chain bench] started worker job sleepy_rod_n64t8m6fyr


Uploading wdladder (0.23 MBs): 100%|██████████| 227726/227726 [00:00<00:00, 1149584.80it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
Uploading wdladder (0.22 MBs): 100%|██████████| 217750/217750 [00:00<00:00, 5016944.69it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


'sleepy_rod_n64t8m6fyr'

In [1]:
import os, sys, glob, shutil, importlib, importlib.util
W = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder"
ALT = "/home/azureuser/cloudfiles/code/Users/estherxin0011/wdladder"
os.chdir(W); sys.path.insert(0, W); importlib.invalidate_caches()
src = next((p for d in (W, ALT) for p in sorted(glob.glob(d + "/chain_v2*.py"))), None)
assert src, "upload chain_v2.py into wdladder first"
shutil.copy(src, W + "/chain.py"); os.remove(src)
import config as C, azure_jobs as AJ
spec = importlib.util.spec_from_file_location("chain", W + "/chain.py")
chain = importlib.util.module_from_spec(spec); sys.modules["chain"] = chain; spec.loader.exec_module(chain)
assert hasattr(chain, "reset"); print("chain loaded", chain.__file__, "v2 OK")
chain.reset("smoke")
base = dict(AJ.stage_runs([1], 120)[0])
chain.add([dict(base, name="chain_smoke_gpu"), dict(base, name="chain_smoke_gpu2")], cid="smoke",
          pool_asset="azureml:wdladder-pool:smoke", extra="--smoke --steps 300 --stop-at-step 150", test_evict=True)
chain.status("bench", runs=False)

chain loaded /mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/chain.py v2 OK
[chain smoke] reset
[chain smoke] +2 runs (queue 2)


Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading wdladder (0.21 MBs): 100%|██

## 0. Setup: code location, packages, secrets
Upload all `.py` files plus `requirements.txt`, `.amlignore` and `tests/smoke_test.py` into
`/home/azureuser/cloudfiles/code/Users/estherxin0011/wdladder/`, keeping the `tests/` sub-folder.
The cell asks for your keys once and stores them in `~/.wdladder_secrets` (chmod 600, never in code).
**Rotate the Foundry key and HF token that were pasted in chat before you start.**

**CHECK BEFORE CONTINUING:** `torch 2.4.1+cpu` is printed and `secrets file OK`.

In [3]:
!pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.0 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.1.2
    Uninstalling pip-26.1.2:
      Successfully uninstalled pip-26.1.2


In [1]:
import os, sys, getpass, importlib
ROOT = "/home/azureuser/cloudfiles/code/Users/estherxin0011"
PROJECT = f"{ROOT}/wdladder"
os.environ["WDL_PROJECT"] = PROJECT
os.environ["WDL_DATA"] = "/home/azureuser/wdladder_data"
os.chdir(PROJECT); sys.path.insert(0, PROJECT)
PY = sys.executable                      # the kernel's own python
print("kernel python:", PY, sys.version.split()[0])
!{PY} -m pip install -q -r requirements.txt
!{PY} -m pip install -q "torch>=2.4" --index-url https://download.pytorch.org/whl/cpu
importlib.invalidate_caches()
import torch; print("torch", torch.__version__)

SEC = os.path.expanduser("~/.wdladder_secrets")
have = open(SEC).read() if os.path.exists(SEC) else ""
lines = [l for l in have.splitlines() if l.strip()]
for k in ("HF_TOKEN", "AZURE_OPENAI_KEY"):
    if f"{k}=" not in have:
        lines.append(f"{k}=" + getpass.getpass(f"{k}: ").strip())
open(SEC, "w").write("\n".join(lines) + "\n"); os.chmod(SEC, 0o600)
import config as C
print("secrets file OK" if C.secret("HF_TOKEN", False) and C.secret("AZURE_OPENAI_KEY", False) else "MISSING SECRET")

kernel python: /anaconda/envs/azureml_py310_sdkv2/bin/python 3.10.20

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: /anaconda/envs/azureml_py310_sdkv2/bin/python -m pip install --upgrade pip
torch 2.14.1+cpu
secrets file OK


## 1. Unit smoke tests (CPU, ~2 min, $0)
These check MAI law values, the Newton-Schulz orthogonality, dropless MoE routing, decay semantics, data-stream
determinism and the analysis pipeline on a synthetic ladder.

**CHECK BEFORE CONTINUING:**
- `ALL SMOKE TESTS PASSED` is printed.
- The analysis selftest recovers a WD ratio close to the true synthetic value (about 1.2).

In [4]:
!python config.py
!python tests/smoke_test.py
!python datastream.py --selftest
!python analysis.py --selftest
!python plan.py --budget

  L     d   H/KV     active       total steps@120  tok@120       LR*     WD* GFLOP/tok
  3   256   4/1       1.89M       14.9M      6930     227M  3.22e-03   0.765     0.026
  4   320   5/1       3.89M       27.6M     14250     467M  2.13e-03   0.562     0.042
  6   512   8/2      15.04M       94.2M     27540    1805M  1.46e-03   0.424     0.121
  9   768  12/3      50.65M      302.1M     92745    6078M  7.27e-04   0.253     0.355
cos {'H_none': 1.0, 'H_int': 1.364, 'H_end': 10.0, 'r': 0.733}
wsd {'H_none': 1.0, 'H_int': 1.046, 'H_end': 10.0, 'r': 0.956}
Traceback (most recent call last):
  File "/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/tests/smoke_test.py", line 121, in <module>
    t_config()
  File "/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/tests/smoke_test.py", line 38, in t_config
    assert len(names) == len(set(names))
AssertionError
datastream selftest OK {'web': {'windows': 288, 'u

## 2. Azure resources: CPU cluster, GPU cluster check, job environment (~10 min first time, $0)
This creates `cpu-rlvr` (F16s_v2, **low priority**, 0 to 6 nodes, scale-down after 10 min idle) and checks `rlvr2`
(T4). It also builds the environment `wdladder-env` (torch 2.4.1 + CUDA 11.8).

**CHECK BEFORE CONTINUING:**
- The GPU cluster shows `min 0`. If min is above 0, the T4 bills 24/7: set it to 0 in the portal.
- Note the GPU **tier**. If it is `low_priority`, set `PRICES["gpu_node"]` to about 0.11 in `config.py`; dedicated is 0.526.
- If you have no low-priority CPU quota, set `AZURE["cpu_low_priority"]=False`. CPU cost rises about 5x, so run more L3 work on the instance instead.
- The environment build succeeds (Azure ML studio > Environments > wdladder-env shows *Succeeded* after the first job).

In [2]:
!{PY} patch_step2.py
import importlib, json, config as C, azure_jobs as AJ, plan, analysis
importlib.reload(C); importlib.reload(AJ); importlib.reload(plan)
AJ.ensure_cpu_cluster()          # read-only now
AJ.check_gpu_cluster()
env = AJ.ensure_env(); print(env.name, env.version)

found cluster cpu-rlvr: Standard_E16s_v3, tier dedicated, min 0, max 6
config.py patched
azure_jobs.py patched (ensure_cpu_cluster is read-only)
tests/smoke_test.py patched

projected spend with CPU $1.008/node-h (placeholder throughput until step 8):
stage                  runs   run-h  $ stage   $ cum
1 L3 gate                 6    50.1    29.06   29.06
2 L3 grid                30   339.9   197.00  226.06
3 L3 confirm+WSD         14   156.0    90.43  316.49
4 L4 GPU                 10    39.3    23.76  340.25
5 L6 GPU tpp60            8   141.7    85.72  425.97
6 L9 GPU (opt)            2   149.1    90.21  516.19

NOTE: dedicated E16s_v3 is ~7x the low-priority F16s_v2 price assumed earlier. If stages 1-3 above push the total past your budget, re-run with --l3-on-gpu (L3 on the T4 is ~$0.74/run) or wait for step 8 benchmarks (real CPU speed may change the picture).
cpu cluster: cpu-rlvr Standard_E16s_v3 tier dedicated min 0 max 6 idle_s 120.0
gpu cluster: rlvr2 Standard_NC4as_T4_v3 t

## 3. Data smoke + local train/resume test (~10 min, $0.2)
This builds a tiny pool, then trains 40 steps straight and separately trains 20 steps, simulates pre-emption and resumes.

**CHECK BEFORE CONTINUING:**
- `6 resume OK: straight X == resumed X` is printed, meaning the resumed run is bit-for-bit consistent.
- `finished run skipped` is printed, so re-running a finished run never costs compute again.

In [3]:
!python data_prep.py --smoke all
!python tests/smoke_test.py --resume

[fetch] web: target 0.03 GB text, have 0.10 GB in 1 shards
[fetch] code: target 0.01 GB text, have 0.21 GB in 1 shards
[fetch] math: target 0.01 GB text, have 0.16 GB in 1 shards
[tokenizer] exists: /home/azureuser/wdladder_data_smoke/tokenizer.json
[tokenize] pool exists: /home/azureuser/wdladder_data_smoke/pool (delete it to rebuild)
[verify] OK  web   train      8.6M (need      2.8M)  eval 0.21M tokens / 200 docs, 3.69 bytes/token
[verify] OK  code  train      6.9M (need      0.6M)  eval 0.40M tokens / 200 docs, 3.06 bytes/token
[verify] OK  math  train      6.0M (need      0.6M)  eval 0.29M tokens / 200 docs, 3.09 bytes/token
[verify] PASS
cross-stage duplicates (expected, deduped at submit): ['L3_t120_cos_const_lr1_p1_s1', 'L3_t120_cos_scaled_lr1_p1_s1']
1 config/plan OK {'lr': 0.0056, 'wd': 1.7607, 'prod': 0.0098}
2 Newton-Schulz OK 0.682 1.042
3 MoE OK ce 9.026 aux 1.001
4 decay semantics OK
datastream selftest OK {'web': {'windows': 288, 'unique_tokens': 18906, 'effective_repea

## 4. Foundry (GPT-6.1) connectivity ($0.01)
GPT-6.1 (`gpt-6.1-sol`) is used **only** for three things: anomaly review after each gate, log triage, and
drafting the Results section with automatic number verification. It never takes a gate decision; all gates are
pre-registered in `config.PREREG`. If you deploy a GPT-5-series model for cheap triage, put its name in
`FOUNDRY["deployment_fast"]`.

**CHECK BEFORE CONTINUING:** the reply contains `OK`.

In [4]:
import llm_review
print(llm_review.chat("Reply with the single word OK.", max_tokens=20))

OK


## 5. Full data pool (~1-2.5 h on 20 vCPU, about $3 of instance time)
This builds about 1.27B unique tokens from FineWeb-Edu, The Stack (Python) and OpenWebMath. The pool size
comes from `plan.pool_requirements()` so that **source scaling** (MAI) holds: every rung repeats each source
the same number of times. Run it in a terminal so the browser can close:
```
cd ~/cloudfiles/code/Users/estherxin0011/wdladder
export WDL_DATA=/home/azureuser/wdladder_data
nohup python data_prep.py all > ~/data_prep.log 2>&1 &
tail -f ~/data_prep.log
```
It is resumable: if the instance stops, run the same command again.

**CHECK BEFORE CONTINUING:**
- Every source shows `OK` and the run ends with `[verify] PASS`.
- If any source is `LOW`, run `python data_prep.py fetch --scale 1.2`, delete `pool/`, then run `python data_prep.py tokenize`.
- Bytes per token are about 3.5-4.5 for web and code.

In [10]:
import subprocess, os
DATA = "/home/azureuser/wdladder_data"
LOG, PIDF = os.path.expanduser("~/data_prep.log"), os.path.expanduser("~/data_prep.pid")

def alive():
    try:
        pid = int(open(PIDF).read()); os.kill(pid, 0); return pid
    except Exception:
        return None

if alive():
    print("already running, pid", alive())
else:
    env = dict(os.environ, WDL_DATA=DATA, WDL_PROJECT=PROJECT, PYTHONUNBUFFERED="1")
    p = subprocess.Popen([PY, "data_prep.py", "all"], cwd=PROJECT, env=env,
                         stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True)
    open(PIDF, "w").write(str(p.pid))
    print("started pid", p.pid, "log:", LOG)

started pid 29830 log: /home/azureuser/data_prep.log


In [31]:
pid = alive()
print("RUNNING, pid" if pid else "NOT RUNNING (finished or stopped)", pid or "")
!tail -n 15 ~/data_prep.log
!du -sh /home/azureuser/wdladder_data/* 2>/dev/null; df -h /home/azureuser | tail -1

RUNNING, pid 29830
  web shard 40 3.88/5.68 GB
  web shard 50 4.84/5.68 GB
[fetch] code: target 0.47 GB text, have 0.00 GB in 0 shards
[fetch] math: target 0.27 GB text, have 0.00 GB in 0 shards
[tokenizer] vocab 8192 trained in 14s
[tokenize] web: 1100574 docs 1178.0M tokens (319s)
[tokenize] code: 45284 docs 84.9M tokens (42s)
[tokenize] math: 39695 docs 59.0M tokens (22s)
[tokenize] pool written: /home/azureuser/wdladder_data/pool
[verify] OK  web   train   1174.8M (need   1085.0M)  eval 3.21M tokens / 3000 docs, 3.84 bytes/token
[verify] LOW code  train     79.3M (need    113.0M)  eval 5.58M tokens / 3000 docs, 3.13 bytes/token
[verify] LOW math  train     54.4M (need     67.8M)  eval 4.57M tokens / 3000 docs, 3.20 bytes/token
[verify] FAIL - re-run fetch with --scale 1.2 then delete pool/ and tokenize
2.5G	/home/azureuser/wdladder_data/pool
6.3G	/home/azureuser/wdladder_data/raw
532K	/home/azureuser/wdladder_data/tokenizer.json
/dev/root       119G  108G   11G  91% /


In [32]:
print("still running" if alive() else "finished - OK to continue")

still running


In [17]:
!{PY} data_prep.py verify

Traceback (most recent call last):
  File "/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/data_prep.py", line 207, in <module>
    raise SystemExit(0 if verify(a.smoke, a.scale) else 1)
  File "/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/data_prep.py", line 178, in verify
    pool = datastream.Pool(P["pool"])
  File "/mnt/batch/tasks/shared/LS_root/mounts/clusters/cpurlvr/code/Users/estherxin0011/wdladder/datastream.py", line 34, in __init__
    self.meta = json.load(open(os.path.join(pool_dir, "meta.json")))
FileNotFoundError: [Errno 2] No such file or directory: '/home/azureuser/wdladder_data/pool/meta.json'


In [33]:
import importlib, os, data_prep as D
os.environ["WDL_DATA"] = "/home/azureuser/wdladder_data"
importlib.reload(D)
_orig = D.requirements
BOOST = {"code": 1.6, "math": 1.4}            # 1.43x / 1.25x needed + margin
def _req(smoke, scale):
    r = dict(_orig(smoke, scale))
    for k, f in BOOST.items():
        if isinstance(r.get(k), (int, float)):
            r[k] = r[k] * f
    return r
print("old targets:", _orig(False, 1.0)); D.requirements = _req; print("new targets:", _req(False, 1.0))
D.fetch(False, 1.0)                           # web: target already met -> skipped
D.requirements = _orig

old targets: {'web': 1171834490, 'code': 122066092, 'math': 73239655}
new targets: {'web': 1171834490, 'code': 195305747.20000002, 'math': 102535517.0}


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
import glob, json, hashlib
for src in ("code", "math"):
    seen, dup, n = set(), 0, 0
    for f in sorted(glob.glob(f"/home/azureuser/wdladder_data/raw/{src}/shard_*.jsonl")):
        for line in open(f):
            d = json.loads(line); t = d.get("text") or d.get("content") or line
            h = hashlib.md5(t.encode()).digest(); n += 1
            if h in seen: dup += 1
            seen.add(h)
    print(f"{src}: {n} docs, {dup} duplicates ({100*dup/max(n,1):.2f}%)")

code: 80000 docs, 0 duplicates (0.00%)
math: 60000 docs, 537 duplicates (0.90%)


In [35]:
!rm -rf /home/azureuser/wdladder_data/pool
!WDL_DATA=/home/azureuser/wdladder_data {PY} data_prep.py tokenize
!WDL_DATA=/home/azureuser/wdladder_data {PY} data_prep.py verify
!df -h /home/azureuser | tail -1

[tokenize] web: 1100574 docs 1178.0M tokens (322s)
[tokenize] code: 58891 docs 110.0M tokens (57s)
[tokenize] math: 56331 docs 83.7M tokens (32s)
[tokenize] pool written: /home/azureuser/wdladder_data/pool
[verify] OK  web   train   1174.8M (need   1085.0M)  eval 3.21M tokens / 3000 docs, 3.84 bytes/token
[verify] LOW code  train    104.4M (need    113.0M)  eval 5.61M tokens / 3000 docs, 3.21 bytes/token
[verify] OK  math  train     79.3M (need     67.8M)  eval 4.41M tokens / 3000 docs, 3.18 bytes/token
[verify] FAIL - re-run fetch with --scale 1.2 then delete pool/ and tokenize
/dev/root       119G  108G   11G  92% /


In [36]:
import os, shutil, subprocess, importlib, data_prep as D, config as C, azure_jobs as AJ
DATA = "/home/azureuser/wdladder_data"; os.environ["WDL_DATA"] = DATA
importlib.reload(C); importlib.reload(D); importlib.reload(AJ)

_orig = D.requirements
def _req(smoke, scale):
    r = dict(_orig(smoke, scale)); r["code"] *= 2.1; r["math"] *= 1.4; return r
D.requirements = _req
print("targets:", _req(False, 1.0))
D.fetch(False, 1.0)
D.requirements = _orig

shutil.rmtree(f"{DATA}/pool", ignore_errors=True)
env = dict(os.environ, WDL_DATA=DATA, PYTHONUNBUFFERED="1")
subprocess.run([PY, "data_prep.py", "tokenize"], env=env, check=True)
ok = subprocess.run([PY, "data_prep.py", "verify"], env=env).returncode == 0
!df -h /home/azureuser | tail -1
assert ok, "verify still FAIL - send me the output"

print(AJ.upload_pool(C.data_paths(C.SMOKE_ROOT)["pool"]))
print(AJ.upload_pool(C.P["pool"]))
ml = AJ.client()
for v in ("smoke", "1"):
    d = ml.data.get("wdladder-pool", version=v); print("asset OK:", d.name, d.version)

targets: {'web': 1171834490, 'code': 256338793.20000002, 'math': 102535517.0}
[fetch] web: target 5.68 GB text, have 5.71 GB in 59 shards
[fetch] code: target 0.97 GB text, have 0.84 GB in 4 shards
[fetch] math: target 0.37 GB text, have 0.48 GB in 3 shards
[tokenize] web: 1100574 docs 1178.0M tokens (324s)
[tokenize] code: 68432 docs 127.6M tokens (68s)
[tokenize] math: 56331 docs 83.7M tokens (33s)
[tokenize] pool written: /home/azureuser/wdladder_data/pool
creation_context:
  created_at: '2026-10-21T11:06:46.121540+00:00'
  created_by: esther xin
  created_by_type: User
  last_modified_at: '2026-10-21T11:06:46.133358+00:00'
description: 'WDLadder token pool {''web_eval'': 208304, ''web_train'': 8609960, ''code_eval'':
  396708, ''code_train'': 6855532, ''math_eval'': 288441, ''math_train'': 5965407}'
id: /subscriptions/7a8513b6-ada2-4ad1-aee2-687fa5663c82/resourceGroups/AIModels/providers/Microsoft.MachineLearningServices/workspaces/RLVR/data/wdladder-pool/versions/smoke
name: wdlad

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Uploading pool (44.81 MBs): 100%|██████████| 44810356/44810356 [00:00<00:00, 210072691.09it/s]


Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy '/home/azureuser/wdladder_data/pool' 'https://rlvr1860064645.blob.core.windows.net/azureml-blobstore-5afa9458-1bb4-40c4-b854-5648c5d7439a/LocalUpload/41c8191911f4fcc20257db75dc881b31078ec6b7f525418c463d0c155e127473/pool' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more informat

In [ ]:
!python data_prep.py verify

## 6. Upload pools as data assets (~10 min, $0)
Jobs **download** the pool to node-local disk, so training never reads from network storage.

**CHECK BEFORE CONTINUING:** both assets are listed: `wdladder-pool:smoke` and `wdladder-pool:1`.

In [ ]:
AJ.upload_pool(C.data_paths(C.SMOKE_ROOT)["pool"])
AJ.upload_pool(C.P["pool"])

## 7. Cluster smoke: pre-emption and resume on the real CPU and GPU clusters (~30 min, ~$0.3)
Segment 1 stops at step 150 (exit 75, so Azure marks the job *Failed*; **this is expected**). Segment 2 is a
resubmission that must resume from the checkpoint on the datastore and finish.

**CHECK BEFORE CONTINUING:**
- Both runs print `status done, segments 2`.
- The GPU log shows `device cuda` and a finite bpb.
- Job snapshot size is under 1 MB. If it is larger, `.amlignore` was not uploaded.

In [37]:
import time, json, importlib, config as C, azure_jobs as AJ, plan
importlib.reload(C); importlib.reload(AJ); importlib.reload(plan)
ml = AJ.client()

def wait(jobs, poll=60):
    while True:
        st = {j: ml.jobs.get(j).status for j in jobs}
        print(time.strftime("%H:%M"), st, flush=True)
        if all(s in ("Completed", "Failed", "Canceled") for s in st.values()):
            return st
        time.sleep(poll)

def adhoc(name, device, L=3):
    return dict(name=name, stage=0, L=L, tpp=120, sched="cos", method="scaled",
                lr_mult=1.0, prod_mult=1.0, seed=0, device=device)

# ---- STEP 7: cluster smoke (segment 1 stops at step 150 -> job shows Failed = expected; segment 2 resumes)
SMOKE = [adhoc("smoke_cpu", "cpu"), adhoc("smoke_gpu", "gpu")]
P = "azureml:wdladder-pool:smoke"
wait(AJ.submit(SMOKE, pool_asset=P, extra_args="--smoke --steps 300 --stop-at-step 150", usd_override=0.05, tag_stage="smoke"))
wait(AJ.submit(SMOKE, pool_asset=P, extra_args="--smoke --steps 300", usd_override=0.05, tag_stage="smoke"))
ok = True
for r in SMOKE:
    s = AJ.read_remote_json(r["name"], "summary.json") or {}
    print(r["name"], "| status", s.get("status"), "| segments", s.get("segments"), "| device", s.get("device"),
          "| bpb", (s.get("final") or {}).get("bpb"))
    ok &= s.get("status") == "done" and s.get("segments") == 2
assert ok, "STEP 7 FAILED - open the job in studio > Jobs > Outputs+logs > user_logs/std_log.txt and send it to me"

# ---- STEP 8: throughput benchmarks -> real cost table
BENCH = [adhoc("bench_cpu_L3", "cpu", 3), adhoc("bench_cpu_L4", "cpu", 4),
         adhoc("bench_gpu_L3", "gpu", 3), adhoc("bench_gpu_L4", "gpu", 4),
         adhoc("bench_gpu_L6", "gpu", 6), adhoc("bench_gpu_L9", "gpu", 9)]
wait(AJ.submit(BENCH, extra_args="--benchmark 20", usd_override=0.08, tag_stage="bench"))
AJ.sync()
tps = plan.load_bench()
print("measured tok/s:", {f"{d}-L{L}": round(v) for (d, L), v in sorted(tps.items())})
missing = [b["name"] for b in BENCH if (("gpu" if "gpu" in b["name"] else "cpu"), b["L"]) not in tps]
print("missing benchmarks:", missing or "none")
!{PY} plan.py --budget --l6-tpp 120
!{PY} plan.py --budget --l6-tpp 60

Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Uploading wdladder (0.18 MBs): 100%|██████████| 183789/183789 [00:00<00:00, 4372026.33it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
Uploading wdladder (0.19 MBs): 100%|██████████| 187346/187346 [00:00<00:00, 4681533.75it/s]


pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored
Uploading wdladder (0.19 MBs): 100%|██████████| 189559/189559 [00:00<00:00, 5

[budget] spent $0.00 + in-flight $0.00 + this batch $0.10 = $0.10 / $140
[submit] bright_rice_fxy0jh98tn cpu ['smoke_cpu']
[submit] gifted_dinner_jcgt3tprlg gpu ['smoke_gpu']
11:12 {'bright_rice_fxy0jh98tn': 'Preparing', 'gifted_dinner_jcgt3tprlg': 'Starting'}
11:13 {'bright_rice_fxy0jh98tn': 'Queued', 'gifted_dinner_jcgt3tprlg': 'Queued'}
11:14 {'bright_rice_fxy0jh98tn': 'Queued', 'gifted_dinner_jcgt3tprlg': 'Queued'}
11:15 {'bright_rice_fxy0jh98tn': 'Queued', 'gifted_dinner_jcgt3tprlg': 'Queued'}
11:16 {'bright_rice_fxy0jh98tn': 'Running', 'gifted_dinner_jcgt3tprlg': 'Running'}
11:17 {'bright_rice_fxy0jh98tn': 'Running', 'gifted_dinner_jcgt3tprlg': 'Running'}
11:18 {'bright_rice_fxy0jh98tn': 'Running', 'gifted_dinner_jcgt3tprlg': 'Running'}
11:19 {'bright_rice_fxy0jh98tn': 'Running', 'gifted_dinner_jcgt3tprlg': 'Running'}
11:20 {'bright_rice_fxy0jh98tn': 'Running', 'gifted_dinner_jcgt3tprlg': 'Running'}
11:21 {'bright_rice_fxy0jh98tn': 'Failed', 'gifted_dinner_jcgt3tprlg': 'Running'}

In [ ]:
import time
def wait(jobs, poll=60):
    ml = AJ.client()
    while True:
        st = {j: ml.jobs.get(j).status for j in jobs}
        print(time.strftime("%H:%M"), st, flush=True)
        if all(s in ("Completed", "Failed", "Canceled") for s in st.values()):
            return st
        time.sleep(poll)

def adhoc(name, device, L=3, stage=0):
    return dict(name=name, stage=stage, L=L, tpp=120, sched="cos", method="scaled",
                lr_mult=1.0, prod_mult=1.0, seed=0, device=device)

SMOKE = [adhoc("smoke_cpu", "cpu"), adhoc("smoke_gpu", "gpu")]
j1 = AJ.submit(SMOKE, pool_asset="azureml:wdladder-pool:smoke", extra_args="--smoke --steps 300 --stop-at-step 150",
               usd_override=0.05, tag_stage="smoke")
wait(j1)
j2 = AJ.submit(SMOKE, pool_asset="azureml:wdladder-pool:smoke", extra_args="--smoke --steps 300",
               usd_override=0.05, tag_stage="smoke")
wait(j2)
for r in SMOKE:
    s = AJ.read_remote_json(r["name"], "summary.json") or {}
    print(r["name"], "status", s.get("status"), "segments", s.get("segments"), "device", s.get("device"),
          "bpb", (s.get("final") or {}).get("bpb"))

## 8. Throughput benchmarks -> real budget table (~20 min, ~$0.5)
This measures tokens/s for each rung on each device. `plan.py --budget` then replaces the placeholders with real
numbers.

**CHECK BEFORE CONTINUING (the most important cost check):**
- Every `bench_*` result is present.
- The `$ cum` column at stage 4 is within your budget.
- Stage 5 at TPP 60 fits the remaining budget; otherwise use `--l6-tpp 30` (`L6_TPP` below).
- Stage 6 (L9) is optional. Run it only if the remaining budget is more than its row.
- If CPU L4 is cheaper than GPU L4 per run, set `C.PLACEMENT[4]="cpu"` in config.py (more CPU offload).

In [ ]:
BENCH = [adhoc("bench_cpu_L3", "cpu", 3), adhoc("bench_cpu_L4", "cpu", 4),
         adhoc("bench_gpu_L3", "gpu", 3), adhoc("bench_gpu_L4", "gpu", 4),
         adhoc("bench_gpu_L6", "gpu", 6), adhoc("bench_gpu_L9", "gpu", 9)]
wait(AJ.submit(BENCH, extra_args="--benchmark 20", usd_override=0.08, tag_stage="bench"))
AJ.sync()
!python plan.py --budget --l6-tpp 60
!python plan.py --budget --l6-tpp 30

In [ ]:
L6_TPP = 60          # <- set after reading the table (60 preferred, 30 if budget is tight)

## 9. Stage 1 (L3, CPU cluster): Muon-SW vs constant decay at MAI's law, 2 seeds + TPP30 (~$4)
Six runs (3 nodes x 2 runs). Each run checkpoints every 15 min and on SIGTERM, and resumes automatically when
resubmitted.

**CHECK:** six runs are listed as `active:`.

In [ ]:
AJ.sync()
AJ.submit(plan.stage1(), tag_stage=1)
AJ.status(plan.stage1())

### 9b. Check-in cell: run once or twice a day while any stage is in flight (~1 min)
This syncs results, resubmits pre-empted or crashed runs (they resume from their last checkpoint), and prints
spend. Use the same cell for later stages by changing `STAGES`.
If you keep the instance on anyway, you can instead run the watchdog in a terminal:
`nohup python azure_jobs.py supervise --stages 1 --interval 20 > ~/sup.log 2>&1 &`

**Early-kill check (saves money):**
- `interim_gap` shows the paired const-minus-scaled BPB at matched steps.
- If the gap is negative at 50% and 70% of training for **both** seeds, Gate 1 will FAIL.
- In that case run `AJ.cancel(1)` and go to step 10 with what has finished. The checkpoints are kept.

In [ ]:
STAGES = [1]
AJ.supervise(STAGES, l6_tpp=L6_TPP, once=True)
rows = analysis.load_runs()
print("interim gap (const - scaled bpb) by seed/step:", analysis.interim_gap(rows))
lg = plan.ledger_load(); print("spend so far:", plan.actual_spend(instance_hours=lg.get("instance_hours", 0)))

## 10. GATE 1 (automatic, pre-registered) + GPT-6.1 anomaly review
| Decision | Meaning | Next |
|---|---|---|
| **PASS** | Muon-SW beats constant decay by more than max(0.003 bpb, 2 x seed noise) | stage 2 + stage 4 |
| **WEAK** | gain is positive but inside noise | stage 2 only; no GPU until gate 2 |
| **FAIL** | no gain at MAI hyperparameters | **STOP.** Write the non-replication note (O3) with `analysis --stage 1` + `llm_review draft`; about $10 spent in total |

**CHECK BEFORE CONTINUING:**
- Read `report/report.md`.
- Read the LLM review. It may flag anomalies such as a diverged seed or a gain driven by one seed. Resolve those before spending more.

In [ ]:
AJ.sync()
!python analysis.py --stage 1
!python llm_review.py review --stage 1

## 11. Stage 2 (L3 2-D grid in MAI coordinates, CPU) [+ stage 4 L4 on GPU if PASS] (~$26 + ~$24)
Stage 2 sweeps `(ln LR, ln LR*WD)` as multiples of MAI's law: 3 x 5 points per method. It also adds TPP60 and TPP240
reference runs for the compute-equivalent speed-up law.
Use the check-in cell with `STAGES=[2,4]` (or `[2]` if WEAK).

After stage 2 finishes, `analysis --stage 2` fits 2-D quadratics with bootstrap CIs. **Edge rule:** if an optimum
lies at the grid edge, it writes `edge_ext.json`. Re-running the next cell submits only the extension points.
Repeat at most twice.

**CHECK BEFORE CONTINUING:**
- `edge_extension: []`.
- Both fits are convex: `convex: true`, with R² above 0.9.
- `wd_ratio_cos` has a finite CI.
- The LLM review shows no unresolved issues.

In [ ]:
G1 = json.load(open(os.path.join(C.P["report"], "results.json")))["gate1"]["decision"]
print("gate 1:", G1)
assert G1 in ("PASS", "WEAK"), "gate 1 did not pass - stop here (see step 10)"
STAGES = [2, 4] if G1 == "PASS" else [2]
AJ.sync()
AJ.submit(AJ.stage_runs(STAGES, L6_TPP), tag_stage=2)

In [ ]:
# after all stage-2 runs are done (check-in cell with STAGES above):
AJ.sync()
!python analysis.py --stage 2
!python llm_review.py review --stage 2
ext = json.load(open(os.path.join(C.P["report"], "edge_ext.json")))
if ext:
    print("edge extension needed:", ext); AJ.submit(AJ.stage_runs([2], L6_TPP), tag_stage=2)

## 12. Stage 3 (L3 confirmation at fitted optima, seeds 1-2, + WSD 1-D) -> GATE 2 (~$12)
The **tuned gap** with its CI decides the paper's headline:
- **O1 genuine:** Muon-SW keeps X% of its gain after both methods are tuned.
- **O2 calibration artifact:** the gain disappears; MAI's law plus a fitted constant multiplier is enough.
- **O4 reversal:** tuned constant decay wins.

All three outcomes are publishable; see the earlier outcome table.

**CHECK BEFORE CONTINUING:**
- `tuned_gap` and `survival_fraction` are printed.
- `hypothesis_verdict` names which of H_none, H_int or H_end is consistent with the measured WD ratio.
- **Spend gate:** continue to stage 5 only if `remaining budget > stage-5 row` in `plan.py --budget`.
  Otherwise write up at L3/L4; the paper still stands.

In [ ]:
AJ.sync()
AJ.submit(AJ.stage_runs([3], L6_TPP), tag_stage=3)
# ... check-in cell with STAGES=[3] until done, then:

In [ ]:
AJ.sync()
!python analysis.py --stage 3
!python llm_review.py review --stage 3
!python plan.py --budget --l6-tpp {L6_TPP}

## 13. Stage 5 (L6 on the T4): does the fitted WD ratio transfer up the ladder? (~$45-86)
This tests whether the stage-2/3 ratio, predicted for L6 from L3 and L4, lands on the L6 optimum. That is the
ladder-transfer claim at the core of MAI's method. Runs are ordered so the most informative points go first; you can
cancel the rest with `AJ.cancel(5)` if the budget runs low.

**CHECK BEFORE CONTINUING:** `L6` in the output shows predicted vs measured optimum with CI, and the budget remains.

In [ ]:
AJ.sync()
AJ.submit(AJ.stage_runs([5], L6_TPP), tag_stage=5)
# check-in cell with STAGES=[5]; then:

In [ ]:
AJ.sync()
!python analysis.py --stage 5
!python llm_review.py review --stage 5

## 14. Optional stage 6 (L9 TPP30, 2 runs on the T4)
Run this only if `plan.py --budget` shows enough remaining budget. It gives one larger-rung point: constant decay
at MAI's law vs Muon-SW at its predicted optimum.

In [ ]:
# AJ.submit(AJ.stage_runs([6], L6_TPP), tag_stage=6)
# later: AJ.sync(); !python analysis.py --stage 6

## 15. Report, draft, spend ledger, cleanup
- `report/report.md`, `report/results.json` and `report/fig_*.png` are the paper's results.
- `report/results_draft.tex` is GPT-6.1's Results draft. Every number in it is checked against results.json, and
  any unverified number is listed at the bottom. **Hand-check them.**
- Log instance hours (Azure portal > Cost analysis, or your own count) so the ledger is complete.

**CHECK:**
- No active jobs remain.
- The GPU cluster has scaled to 0 nodes.
- **The compute instance is stopped.**

In [ ]:
FINAL_STAGE = 5          # highest stage you ran
!python analysis.py --stage {FINAL_STAGE}
!python llm_review.py draft
lg = plan.ledger_load(); lg["instance_hours"] = 0.0   # <- enter total instance hours used
plan.ledger_save(lg)
print(plan.actual_spend(instance_hours=lg["instance_hours"]))
print("active jobs:", AJ.active_runs())